# Robot arm, live testing 03.03.2026

### Connect to the simulator / to the robot

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor

In [ ]:
file_path = "trapezoid_letters-trapezoid_letters-trace.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print(data["generated_at"])

------ simulation only ----------

In [ ]:
from duckify_simulation.duckify_sim import DuckifySim as ISCoin

SIMULATION = True

# # Connect to the simulator
iscoin = ISCoin()
robot = iscoin.robot_control
print("Connected to simulator")

----------- REAL ROBOT ONLY ----------------

In [ ]:
#from URBasic import ISCoin

In [ ]:
# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
if not SIMULATION:
    iscoin = ISCoin(host="10.30.5.158", opened_gripper_size_mm=40)

In [ ]:
robot_arm = iscoin.robot_control

## Default home position

This joint position is equivalent to this one:
```
home_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )
```



In [ ]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)
robot_arm.movej(home)

tcp_home = robot_arm.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

In [ ]:
# target_tcp = TCP6D.createFromMetersRadians(
#       0.0,      # x
#      -0.35,     # y
#       0.20,     # z
#       0.0,      # rx
#       3.14,      # ry
#       0.0       # rz
#   )

In [ ]:
# current_joints = robot.get_actual_joint_positions()
# target_joints = robot.get_inverse_kin(target_tcp, qnear=current_joints)

## Set TCP

In [ ]:
if not SIMULATION:
    from src.calibration import collect_data
    NUM_MEASURES = 20

    tcps = collect_data(robot_arm, NUM_MEASURES)
else:
    tcps = [
        [0.0024946337740892888, -0.3594913915339868, 0.06033167700558813, -0.62069755529755, -2.369727486212462, -0.13752393611863362],
        [-0.07482225018547103, -0.35685269130655106, 0.07009337653443365, 1.9372057315922462, 2.190232377619723, 0.6719840297817087],
        [-0.08776793211801745, -0.2979812165503483, 0.051154992555969794, 2.3781388566898687, 0.2623998383341909, 0.7178051284770465],
        [-0.0006933205535983589, -0.340924257935631, 0.06551204074775754, -2.879088789274111, -0.4716877213448937, 0.8276834472055694],
        [-0.04094004360899719, -0.3437262194580261, 0.0786140415986546, -3.087426800813378, 0.2627406174702693, 0.06421242457662117],
        [0.026621421400990636, -0.3469050147213359, 0.033956680032955974, -2.5014091646201537, -0.22918589994059693, 1.5168173293250897],
    ]

In [ ]:
from src.calibration import get_tcp_offset, validate_calibration
tcp_offset = get_tcp_offset(tcps)
robot_arm.set_tcp(tcp_offset)
if not SIMULATION:
    robot_arm.freedrive_mode()
    input()
    robot_arm.end_freedrive_mode()
    validate_calibration(robot_arm)

In [35]:
def normal_to_rxyz(n):
    x,z,y = n
    z = z
    rx = np.arctan2(-y, np.sqrt(x**2 + z**2))
    ry = np.arctan2(x, z)
    rz = -1.4319116960396927e-05
    return [rx,ry,rz]

In [36]:
def create_transformation(A, B):
    A = np.asarray(A)[:, :3]
    B = np.asarray(B)[:, :3]
    assert A.shape == B.shape
    N = A.shape[0]

    # Build homogeneous source matrix
    P = np.hstack([A, np.ones((N, 1))])  # Nx4

    # Solve P @ T = B  → T is 4×3
    T, _, _, _ = np.linalg.lstsq(P, B, rcond=None)

    # Extract affine components
    R = T[:3, :].T      # 3×3 linear part
    t = T[3, :]         # 3-vector translation

    # Precompute normal transform
    R_normal = np.linalg.inv(R).T

    def AtoB(p):
        p = np.asarray(p)
        point = p[:3]
        normal = p[3:]

        # Transform point
        p_new = R @ point + t

        # Transform normal
        n_new = R_normal @ normal
        n_new /= np.linalg.norm(n_new)

        # Convert to your rxyz representation
        r_new = normal_to_rxyz(n_new)

        return (*p_new, *r_new)

    return AtoB

In [37]:
from src.transformation import collect_data as collect_data_tf
p_world = np.array(data["calibration"])
if len(p_world) < 4:
    print("Missing point")
if not SIMULATION:
    p_tcp = collect_data_tf(robot_arm, p_world)
else:
    p_tcp = np.array([
        [-0.030, -0.285, 0.180],
        [-0.030, -0.285, 0.220],
        [ 0.030, -0.285, 0.220],
        [ 0.060, -0.25,  0.160]
    ])

world2tcp = create_transformation(p_world, p_tcp)

Generate trajectory

In [ ]:
from src.transformation import tcp_trans
traces = data["traces"]
draw_motions = []
move_motions = []
above = [0,0,-0.02,0,0,0]
above_ = [0,0,0.02,0,0,0]
for trace in traces:
    normal = trace["face"]
    normal[2] = - normal[2]
    t_world = trace["path"]
    motion = []
    buffer_motion = []
    
    # Drawing motion
    for i, p in enumerate(t_world):
        p_w = (*p, *normal)
        x,y,z,rx,ry,rz = world2tcp(p_w)
        
        # Add entry point
        if i == 0:
            x_t,y_t,z_t,rx_t,ry_t,rz_t = tcp_trans([x,y,z,rx,ry,rz], above)
            t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
            m = TCP6DDescriptor(t)
            buffer_motion.append(m)

        # Add drawing point
        t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
        m = TCP6DDescriptor(t)
        buffer_motion.append(m)

        # Add exit point
        if i == len(t_world)-1:
            x_t,y_t,z_t,rx_t,ry_t,rz_t = tcp_trans([x,y,z,rx,ry,rz], above_)
            t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
            m = TCP6DDescriptor(t)
            buffer_motion.append(m)
        if len(buffer_motion) >= 20:
            motion.append(buffer_motion)
            buffer_motion = []
    if buffer_motion:
        motion.append(buffer_motion)
    if motion:
        draw_motions.append(motion)
        move_motions.append((motion[0][0], motion[-1][-1]))
    #break

In [39]:
move_outside = []
for m in range(len(move_motions)):
    if m == 0:
        move_outside.append([TCP6DDescriptor(tcp_home), move_motions[m][0]])
    else:
        move_outside.append([move_motions[m-1][-1], TCP6DDescriptor(tcp_home), move_motions[m][0]])
move_outside.append([move_motions[-1][-1], TCP6DDescriptor(tcp_home)])

In [40]:
move_outside = []
for m in range(len(move_motions)):
    if m == 0:
        move_outside.append([home, move_motions[m][0].tcp])
    else:
        move_outside.append([move_motions[m-1][-1].tcp, home, move_motions[m][0].tcp])
move_outside.append([move_motions[-1][-1].tcp, home])

### Drawing algorithm live

In [41]:
robot_arm.movej(home)
for k in range(len(move_motions)):
    #py.move(move_outside[k])
    
    for m in move_outside[k]:
        if isinstance(m, TCP6D):
            print(m)
            robot_arm.movel(m)
        elif isinstance(m, Joint6D):
            print(robot_arm.get_actual_tcp_pose())
            robot_arm.movej(m)
    input()

    print()

    #for m in draw_motions[k]:
    #    robot_arm.movel_waypoints(m)
        

for m in move_outside[-1]:
    if isinstance(m, TCP6D):
        robot_arm.movel(m)
        print(m)
    elif isinstance(m, Joint6D):
        robot_arm.movej(m)
        print(robot_arm.get_actual_tcp_pose())


movej sent (duration=3s)
Movement OK — target reached
TCP6D([0.0006160330416391965, -0.350407626063329, 0.2653366127630044, -5.066447371099954e-06, 3.139981903281389, -1.4319116960396927e-05])
movej sent (duration=3s)
Movement OK — target reached
TCP6D([-0.010546875000000042, -0.28499961723491785, 0.206178950335594, -1.1753271966785698e-14, -3.141592653589785, -1.4319116960396927e-05])
movej sent (duration=2s)
Movement OK — target reached

TCP6D([-0.010546875000000044, -0.2849996172349179, 0.20173956645161856, -1.1753271966785698e-14, -3.141592653589785, -1.4319116960396927e-05])
movej sent (duration=2s)
Movement OK — target reached
TCP6D([-0.010546874999620358, -0.2849996172342595, 0.2017395664505521, 0.0, 3.14159265355716, 1.4319117487694152e-05])
movej sent (duration=3s)
Movement OK — target reached
TCP6D([-0.009531250000000036, -0.27395987489168594, 0.1736913595861232, 1.051650190031454, 3.1415926535897927, -1.4319116960396927e-05])
movej sent (duration=2s)
Movement OK — target rea